## 1.0 Set up File

1.1 Import packages

In [1]:
import pandas as pd
from PIL import Image
import numpy as np
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf  # For tf.data
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

1.2 Define Model Parameters

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 64

1.3 Read in Data

In [3]:
# Load dataframe
df = pd.read_csv("fossils_augmented.csv")

# Shuffle the dataframe for tensorflow training
df = df.sample(frac=1, random_state=10).reset_index(drop=True)

In [4]:
print(f"Total images: {len(df)}")
print(df["period"].value_counts())

Total images: 3611
period
Silurian Period         1024
Ordovician Period        337
Permian Period           250
Carboniferous Period     250
Jurassic Period          250
Cretaceous Period        250
Triassic Period          250
Cambrian Period          250
Precambrian Eon          250
Devonian Period          250
Cenozoic Era             250
Name: count, dtype: int64


1.4 Train/Test Split Data

In [5]:
# Split so no augmented images are in validation/test sets
train_df = df[df["augmented"] == True]
orig_df  = df[df["augmented"] == False]

# 80/20 split on originals only
val_df   = orig_df.sample(frac=0.2, random_state=42)
train_orig_df = orig_df.drop(val_df.index)

# Combine original training rows with augmented rows
train_df = pd.concat([train_orig_df, train_df], ignore_index=True).sample(frac=1, random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 3144 | Val: 467


1.5 Run DataGenerators on Images

In [ ]:
train_datagen = ImageDataGenerator()
val_datagen   = ImageDataGenerator()

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col="local_path",
    y_col="period",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical", # one-hot encoding period labels
    shuffle=True,
    seed=1
)

val_gen = val_datagen.flow_from_dataframe(
    val_df,
    x_col="local_path",
    y_col="period",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical", # one-hot encoding period labels
    shuffle=False
)

Found 3144 validated image filenames belonging to 11 classes.
Found 467 validated image filenames belonging to 11 classes.


1.6 Save Class Names and Batches

In [7]:
# Save class names for prediction later
import json
class_names = list(train_gen.class_indices.keys())
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print(f"\nClasses: {class_names}")
print(f"Training batches:   {len(train_gen)}")
print(f"Validation batches: {len(val_gen)}")


Classes: ['Cambrian Period', 'Carboniferous Period', 'Cenozoic Era', 'Cretaceous Period', 'Devonian Period', 'Jurassic Period', 'Ordovician Period', 'Permian Period', 'Precambrian Eon', 'Silurian Period', 'Triassic Period']
Training batches:   50
Validation batches: 8


## 2.0 Create Model

2.1 Define EfficientNetB0 Model

In [8]:
# Number of classes for final layer
NUM_CLASSES = len(class_names)

# build model
base = EfficientNetB0(
    weights="imagenet",      # use pretrained weights
    include_top=False,       # remove ImageNet classification head
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False       # freeze base for phase 1

# add custom classification head
model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])


# HEAD ONLY TRAINING is for the dense, final output layers that are added on the frozen base
# BASE layers are those that extract features from images and are pretrained on ImageNet
# compile model for training
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3), # higher learning rate for head-only training
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         2,827 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385,454 (16.73 MB)

 Trainable params: 333,323 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

2.2 Head Only Training

In [ ]:
# Train the head layers first with the base frozen to get a good starting point for the final layers
hist1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10, # number of forward/backward passes on the whole dataset, can be adjusted based on convergence
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ]
)

2.3 Unfreeze Base, Whole Model Training

In [ ]:
# UNFREEZE BASE for fine-tuning with a much lower learning rate to avoid destroying pretrained weights
base.trainable = True

# Recompile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lower learning rate
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Fine-tune the whole model with a lower learning rate and more epochs, using callbacks to prevent overfitting
hist2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30, # number of forward/backward passes on the whole dataset, can be adjusted based on convergence
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.2)
    ]
)

# save the final model for later prediction
model.save("fossil_model.keras")